# Run Any Kind of OLS Regression (ANOVA, GLM, etc.)

### Authors: Calvin Howard.

#### Last updated: July 6, 2023

Use this to run/test a statistical model (e.g., regression or T-tests) on a spreadsheet.

Notes:
- To best use this notebook, you should be familar with GLM design and Contrast Matrix design. See this webpage to get started:
[FSL's GLM page](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/GLM)

# 00 - Import CSV with All Data
**The CSV is expected to be in this format**
- ID and absolute paths to niftis are critical
```
+-----+----------------------------+--------------+--------------+--------------+
| ID  | Nifti_File_Path            | Covariate_1  | Covariate_2  | Covariate_3  |
+-----+----------------------------+--------------+--------------+--------------+
| 1   | /path/to/file1.nii.gz      | 0.5          | 1.2          | 3.4          |
| 2   | /path/to/file2.nii.gz      | 0.7          | 1.4          | 3.1          |
| 3   | /path/to/file3.nii.gz      | 0.6          | 1.5          | 3.5          |
| 4   | /path/to/file4.nii.gz      | 0.9          | 1.1          | 3.2          |
| ... | ...                        | ...          | ...          | ...          |
+-----+----------------------------+--------------+--------------+--------------+
```

Prep Output Direction

In [109]:
# Specify where you want to save your results to
out_dir = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/atrophy_and_stim/results/fornix_dbs/memory_on_x'

Import Data

In [110]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Users/cu135/Downloads/master_list_v3.csv'
sheet = None #'master_list_proper_subjects'

In [111]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
data_df

,cohort_key,stimulation_modality,subject_id,subject,delta_adasmem,percent_improvement_adas,age,baseline_adas,followup_adas,atrophy_path,...,queensland_visuospatial_q4__connectivity_path_map_weighted_mean,corbetta_attention__atrophy_path_map_weighted_mean,corbetta_language__atrophy_path_map_weighted_mean,corbetta_motor__atrophy_path_map_weighted_mean,corbetta_verbal_memory__atrophy_path_map_weighted_mean,corbetta_visuospatial__atrophy_path_map_weighted_mean,corbetta_visuospatial_memory__atrophy_path_map_weighted_mean,queensland_executive_q6__atrophy_path_map_weighted_mean,queensland_language_q2__atrophy_path_map_weighted_mean,queensland_visuospatial_q4__atrophy_path_map_weighted_mean
0,fornix_dbs,DBS,sub-101,101.0,NaN,-0.214286,62.0,28.0,34.0,files/fornix_dbs/sub-101/atrophy_path.nii.gz,...,2.206868,-0.114221,-0.259572,-0.086807,0.043012,-0.241508,-0.266231,0.010522,-0.036987,0.108055
1,fornix_dbs,DBS,sub-102,102.0,NaN,-0.363636,77.0,22.0,30.0,files/fornix_dbs/sub-102/atrophy_path.nii.gz,...,2.722033,-0.085618,-0.215098,-0.031658,0.034536,-0.208621,-0.216075,-0.049319,-0.032709,-0.018841
2,fornix_dbs,DBS,sub-103,103.0,NaN,-0.789474,76.0,19.0,34.0,files/fornix_dbs/sub-103/atrophy_path.nii.gz,...,3.045456,-0.135991,-0.237436,-0.091272,0.078643,-0.211949,-0.272902,0.007281,-0.000917,0.054750
3,fornix_dbs,DBS,sub-105,105.0,NaN,-0.105263,50.0,19.0,21.0,files/fornix_dbs/sub-105/atrophy_path.nii.gz,...,1.339978,-0.088176,-0.124926,-0.049305,0.015261,-0.091895,-0.121565,-0.003670,-0.009720,0.009777
4,fornix_dbs,DBS,sub-106,106.0,NaN,-0.384615,66.0,13.0,18.0,files/fornix_dbs/sub-106/atrophy_path.nii.gz,...,2.910447,-0.080701,-0.133136,-0.046285,0.043780,-0.154481,-0.156869,-0.029679,-0.032856,-0.002890
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,bnm,DBS,Pat8,NaN,NaN,-11.111111,NaN,NaN,NaN,files/bnm/Pat8/atrophy_path.nii.gz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,bnm,DBS,Pat9,NaN,NaN,17.500000,NaN,NaN,NaN,files/bnm/Pat9/atrophy_path.nii.gz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,bnm,DBS,Pat4,NaN,NaN,8.330000,NaN,NaN,NaN,files/bnm/Pat4/atrophy_path.nii.gz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
116,bnm,DBS,Pat10,NaN,NaN,33.330000,NaN,NaN,NaN,files/bnm/Pat10/atrophy_path.nii.gz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [112]:
for col in data_df.columns:
    print(col)

cohort_key
stimulation_modality
subject_id
subject
delta_adasmem
percent_improvement_adas
age
baseline_adas
followup_adas
atrophy_path
connectivity_path
efield_path
percent_improvement_memory
memory_network_connectivity
hippocampus_connectivity
rios_network_connectivity
connectivity_damage
memory_network_damage
rios_network_damage
hippocampus_damage
stim_site_damage
data__schaefer50__parcel_01_A
data__schaefer50__parcel_01_FC
data__schaefer50__parcel_02_A
data__schaefer50__parcel_02_FC
data__schaefer50__parcel_03_A
data__schaefer50__parcel_03_FC
data__schaefer50__parcel_04_A
data__schaefer50__parcel_04_FC
data__schaefer50__parcel_05_A
data__schaefer50__parcel_05_FC
data__schaefer50__parcel_06_A
data__schaefer50__parcel_06_FC
data__schaefer50__parcel_07_A
data__schaefer50__parcel_07_FC
data__schaefer50__parcel_08_A
data__schaefer50__parcel_08_FC
data__schaefer50__parcel_09_A
data__schaefer50__parcel_09_FC
data__schaefer50__parcel_10_A
data__schaefer50__parcel_10_FC
data__schaefer50__par

In [113]:
drop_list = ["Month_12_Delta_ADASMemImprove", "queensland_executive_q6__atrophy_path_map_weighted_mean", "memory_network_connectivity"]

In [114]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
data_df

,cohort_key,stimulation_modality,subject_id,subject,delta_adasmem,percent_improvement_adas,age,baseline_adas,followup_adas,atrophy_path,...,queensland_visuospatial_q4__connectivity_path_map_weighted_mean,corbetta_attention__atrophy_path_map_weighted_mean,corbetta_language__atrophy_path_map_weighted_mean,corbetta_motor__atrophy_path_map_weighted_mean,corbetta_verbal_memory__atrophy_path_map_weighted_mean,corbetta_visuospatial__atrophy_path_map_weighted_mean,corbetta_visuospatial_memory__atrophy_path_map_weighted_mean,queensland_executive_q6__atrophy_path_map_weighted_mean,queensland_language_q2__atrophy_path_map_weighted_mean,queensland_visuospatial_q4__atrophy_path_map_weighted_mean
1,fornix_dbs,DBS,sub-102,102.0,NaN,-0.363636,77.0,22.0,30.0,files/fornix_dbs/sub-102/atrophy_path.nii.gz,...,2.722033,-0.085618,-0.215098,-0.031658,0.034536,-0.208621,-0.216075,-0.049319,-0.032709,-0.018841
2,fornix_dbs,DBS,sub-103,103.0,NaN,-0.789474,76.0,19.0,34.0,files/fornix_dbs/sub-103/atrophy_path.nii.gz,...,3.045456,-0.135991,-0.237436,-0.091272,0.078643,-0.211949,-0.272902,0.007281,-0.000917,0.054750
4,fornix_dbs,DBS,sub-106,106.0,NaN,-0.384615,66.0,13.0,18.0,files/fornix_dbs/sub-106/atrophy_path.nii.gz,...,2.910447,-0.080701,-0.133136,-0.046285,0.043780,-0.154481,-0.156869,-0.029679,-0.032856,-0.002890
7,fornix_dbs,DBS,sub-109,109.0,NaN,-0.304348,72.0,23.0,30.0,files/fornix_dbs/sub-109/atrophy_path.nii.gz,...,4.087344,-0.190754,-0.286672,-0.096988,0.114589,-0.259197,-0.327005,0.023763,-0.023570,0.060356
8,fornix_dbs,DBS,sub-110,110.0,NaN,-0.846154,72.0,13.0,24.0,files/fornix_dbs/sub-110/atrophy_path.nii.gz,...,3.438758,-0.174501,-0.249938,-0.112812,0.068019,-0.210358,-0.311075,-0.016422,-0.060485,0.030276
10,fornix_dbs,DBS,sub-113,113.0,NaN,-0.600000,69.0,15.0,24.0,files/fornix_dbs/sub-113/atrophy_path.nii.gz,...,2.699307,-0.097165,-0.201530,-0.064790,0.004889,-0.196664,-0.234098,-0.041791,-0.071156,0.025962
11,fornix_dbs,DBS,sub-114,114.0,NaN,-0.161290,67.0,31.0,36.0,files/fornix_dbs/sub-114/atrophy_path.nii.gz,...,2.898183,-0.096421,-0.179890,-0.069698,0.054851,-0.184416,-0.233820,0.075236,0.029637,0.066037
13,fornix_dbs,DBS,sub-116,116.0,NaN,-0.368421,67.0,19.0,26.0,files/fornix_dbs/sub-116/atrophy_path.nii.gz,...,3.183413,-0.107483,-0.204717,-0.055331,0.056009,-0.189167,-0.218608,-0.000037,0.005203,0.010442
15,fornix_dbs,DBS,sub-119,119.0,NaN,-0.812500,75.0,16.0,29.0,files/fornix_dbs/sub-119/atrophy_path.nii.gz,...,3.528220,-0.152384,-0.304535,-0.065446,0.097319,-0.263163,-0.282284,0.033729,0.045605,0.012387
16,fornix_dbs,DBS,sub-120,120.0,NaN,-0.277778,78.0,18.0,23.0,files/fornix_dbs/sub-120/atrophy_path.nii.gz,...,0.038855,-0.108687,-0.209144,-0.069465,0.079096,-0.213999,-0.239441,-0.014676,-0.028630,0.069659


**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [115]:
data_df.columns

Index(['cohort_key', 'stimulation_modality', 'subject_id', 'subject',
       'delta_adasmem', 'percent_improvement_adas', 'age', 'baseline_adas',
       'followup_adas', 'atrophy_path',
       ...
       'queensland_visuospatial_q4__connectivity_path_map_weighted_mean',
       'corbetta_attention__atrophy_path_map_weighted_mean',
       'corbetta_language__atrophy_path_map_weighted_mean',
       'corbetta_motor__atrophy_path_map_weighted_mean',
       'corbetta_verbal_memory__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial_memory__atrophy_path_map_weighted_mean',
       'queensland_executive_q6__atrophy_path_map_weighted_mean',
       'queensland_language_q2__atrophy_path_map_weighted_mean',
       'queensland_visuospatial_q4__atrophy_path_map_weighted_mean'],
      dtype='object', length=204)

Set the parameters for dropping rows

In [116]:
column = 'cohort_key'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = "fornix_dbs" # The value to drop if T

In [117]:
data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

,cohort_key,stimulation_modality,subject_id,subject,delta_adasmem,percent_improvement_adas,age,baseline_adas,followup_adas,atrophy_path,...,queensland_visuospatial_q4__connectivity_path_map_weighted_mean,corbetta_attention__atrophy_path_map_weighted_mean,corbetta_language__atrophy_path_map_weighted_mean,corbetta_motor__atrophy_path_map_weighted_mean,corbetta_verbal_memory__atrophy_path_map_weighted_mean,corbetta_visuospatial__atrophy_path_map_weighted_mean,corbetta_visuospatial_memory__atrophy_path_map_weighted_mean,queensland_executive_q6__atrophy_path_map_weighted_mean,queensland_language_q2__atrophy_path_map_weighted_mean,queensland_visuospatial_q4__atrophy_path_map_weighted_mean
1,fornix_dbs,DBS,sub-102,102.0,NaN,-0.363636,77.0,22.0,30.0,files/fornix_dbs/sub-102/atrophy_path.nii.gz,...,2.722033,-0.085618,-0.215098,-0.031658,0.034536,-0.208621,-0.216075,-0.049319,-0.032709,-0.018841
2,fornix_dbs,DBS,sub-103,103.0,NaN,-0.789474,76.0,19.0,34.0,files/fornix_dbs/sub-103/atrophy_path.nii.gz,...,3.045456,-0.135991,-0.237436,-0.091272,0.078643,-0.211949,-0.272902,0.007281,-0.000917,0.054750
4,fornix_dbs,DBS,sub-106,106.0,NaN,-0.384615,66.0,13.0,18.0,files/fornix_dbs/sub-106/atrophy_path.nii.gz,...,2.910447,-0.080701,-0.133136,-0.046285,0.043780,-0.154481,-0.156869,-0.029679,-0.032856,-0.002890
7,fornix_dbs,DBS,sub-109,109.0,NaN,-0.304348,72.0,23.0,30.0,files/fornix_dbs/sub-109/atrophy_path.nii.gz,...,4.087344,-0.190754,-0.286672,-0.096988,0.114589,-0.259197,-0.327005,0.023763,-0.023570,0.060356
8,fornix_dbs,DBS,sub-110,110.0,NaN,-0.846154,72.0,13.0,24.0,files/fornix_dbs/sub-110/atrophy_path.nii.gz,...,3.438758,-0.174501,-0.249938,-0.112812,0.068019,-0.210358,-0.311075,-0.016422,-0.060485,0.030276
10,fornix_dbs,DBS,sub-113,113.0,NaN,-0.600000,69.0,15.0,24.0,files/fornix_dbs/sub-113/atrophy_path.nii.gz,...,2.699307,-0.097165,-0.201530,-0.064790,0.004889,-0.196664,-0.234098,-0.041791,-0.071156,0.025962
11,fornix_dbs,DBS,sub-114,114.0,NaN,-0.161290,67.0,31.0,36.0,files/fornix_dbs/sub-114/atrophy_path.nii.gz,...,2.898183,-0.096421,-0.179890,-0.069698,0.054851,-0.184416,-0.233820,0.075236,0.029637,0.066037
13,fornix_dbs,DBS,sub-116,116.0,NaN,-0.368421,67.0,19.0,26.0,files/fornix_dbs/sub-116/atrophy_path.nii.gz,...,3.183413,-0.107483,-0.204717,-0.055331,0.056009,-0.189167,-0.218608,-0.000037,0.005203,0.010442
15,fornix_dbs,DBS,sub-119,119.0,NaN,-0.812500,75.0,16.0,29.0,files/fornix_dbs/sub-119/atrophy_path.nii.gz,...,3.528220,-0.152384,-0.304535,-0.065446,0.097319,-0.263163,-0.282284,0.033729,0.045605,0.012387
16,fornix_dbs,DBS,sub-120,120.0,NaN,-0.277778,78.0,18.0,23.0,files/fornix_dbs/sub-120/atrophy_path.nii.gz,...,0.038855,-0.108687,-0.209144,-0.069465,0.079096,-0.213999,-0.239441,-0.014676,-0.028630,0.069659


**Standardize Data**
- Enter Columns you Don't want to standardize into a list

In [118]:
data_df.columns

Index(['cohort_key', 'stimulation_modality', 'subject_id', 'subject',
       'delta_adasmem', 'percent_improvement_adas', 'age', 'baseline_adas',
       'followup_adas', 'atrophy_path',
       ...
       'queensland_visuospatial_q4__connectivity_path_map_weighted_mean',
       'corbetta_attention__atrophy_path_map_weighted_mean',
       'corbetta_language__atrophy_path_map_weighted_mean',
       'corbetta_motor__atrophy_path_map_weighted_mean',
       'corbetta_verbal_memory__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial_memory__atrophy_path_map_weighted_mean',
       'queensland_executive_q6__atrophy_path_map_weighted_mean',
       'queensland_language_q2__atrophy_path_map_weighted_mean',
       'queensland_visuospatial_q4__atrophy_path_map_weighted_mean'],
      dtype='object', length=204)

In [119]:
# Remove anything you don't want to standardize
cols_not_to_standardize = [] #['Age']
drop_list = ['CorrWithSiddiqiMap', 'depression_raw_baeline', 'depression_raw_follow_up']

In [120]:
data_df = cal_palm.standardize_columns(cols_not_to_standardize)
data_df

Unable to standardize column cohort_key.
Unable to standardize column stimulation_modality.
Unable to standardize column subject_id.
Unable to standardize column atrophy_path.
Unable to standardize column connectivity_path.
Unable to standardize column efield_path.
Unable to standardize column data__yeo7__source_file.
Unable to standardize column stimpyper_optimized_bilateral_vtas.
Unable to standardize column stimpyper_clinician_bilateral_vtas.


,cohort_key,stimulation_modality,subject_id,subject,delta_adasmem,percent_improvement_adas,age,baseline_adas,followup_adas,atrophy_path,...,queensland_visuospatial_q4__connectivity_path_map_weighted_mean,corbetta_attention__atrophy_path_map_weighted_mean,corbetta_language__atrophy_path_map_weighted_mean,corbetta_motor__atrophy_path_map_weighted_mean,corbetta_verbal_memory__atrophy_path_map_weighted_mean,corbetta_visuospatial__atrophy_path_map_weighted_mean,corbetta_visuospatial_memory__atrophy_path_map_weighted_mean,queensland_executive_q6__atrophy_path_map_weighted_mean,queensland_language_q2__atrophy_path_map_weighted_mean,queensland_visuospatial_q4__atrophy_path_map_weighted_mean
1,fornix_dbs,DBS,sub-102,-1.832786,NaN,-0.139559,1.304612,0.751055,0.839372,files/fornix_dbs/sub-102/atrophy_path.nii.gz,...,-0.729169,0.786184,-0.303617,1.641792,-0.038416,-0.295439,0.152551,-1.457882,-0.169345,-1.083544
2,fornix_dbs,DBS,sub-103,-1.761748,NaN,-1.349839,1.036541,0.107294,1.380902,files/fornix_dbs/sub-103/atrophy_path.nii.gz,...,-0.454578,-0.793563,-0.677034,-0.885473,1.297227,-0.339307,-0.830457,0.318892,0.714451,0.302704
4,fornix_dbs,DBS,sub-106,-1.548633,NaN,-0.199184,-1.644168,-1.180229,-0.785219,files/fornix_dbs/sub-106/atrophy_path.nii.gz,...,-0.569202,0.940369,1.066515,1.021708,0.241525,0.418277,1.176707,-0.841343,-0.173439,-0.783065
7,fornix_dbs,DBS,sub-109,-1.335518,NaN,0.028946,-0.035743,0.965642,0.839372,files/fornix_dbs/sub-109/atrophy_path.nii.gz,...,0.429999,-2.511013,-1.500093,-1.127783,2.385714,-0.962162,-1.766361,0.836299,0.084706,0.408299
8,fornix_dbs,DBS,sub-110,-1.264480,NaN,-1.510931,-0.035743,-1.180229,0.027077,files/fornix_dbs/sub-110/atrophy_path.nii.gz,...,-0.120659,-2.001314,-0.886022,-1.798611,0.975500,-0.318334,-1.490784,-0.425195,-0.941482,-0.158310
10,fornix_dbs,DBS,sub-113,-1.051365,NaN,-0.811333,-0.839956,-0.751055,0.027077,files/fornix_dbs/sub-113/atrophy_path.nii.gz,...,-0.748463,0.424065,-0.076806,0.237204,-0.936155,-0.137815,-0.159224,-1.221556,-1.238124,-0.239584
11,fornix_dbs,DBS,sub-114,-0.980327,NaN,0.435532,-1.376097,2.682338,1.651668,files/fornix_dbs/sub-114/atrophy_path.nii.gz,...,-0.579615,0.447399,0.284939,0.029150,0.576748,0.023649,-0.154403,2.452118,1.563818,0.515318
13,fornix_dbs,DBS,sub-116,-0.838251,NaN,-0.153158,-1.376097,0.107294,0.297842,files/fornix_dbs/sub-116/atrophy_path.nii.gz,...,-0.337450,0.100456,-0.130083,0.638197,0.611811,-0.038978,0.108736,0.089177,0.884571,-0.531944
15,fornix_dbs,DBS,sub-119,-0.625136,NaN,-1.415283,0.768470,-0.536468,0.703989,files/fornix_dbs/sub-119/atrophy_path.nii.gz,...,-0.044704,-1.307678,-1.798707,0.209375,1.862739,-1.014456,-0.992750,1.149142,2.007701,-0.495298
16,fornix_dbs,DBS,sub-120,-0.554098,NaN,0.104461,1.572683,-0.107294,-0.108306,files/fornix_dbs/sub-120/atrophy_path.nii.gz,...,-3.007225,0.062701,-0.204091,0.038999,1.310922,-0.366329,-0.251644,-0.370362,-0.055937,0.583557


In [121]:
data_df.columns

Index(['cohort_key', 'stimulation_modality', 'subject_id', 'subject',
       'delta_adasmem', 'percent_improvement_adas', 'age', 'baseline_adas',
       'followup_adas', 'atrophy_path',
       ...
       'queensland_visuospatial_q4__connectivity_path_map_weighted_mean',
       'corbetta_attention__atrophy_path_map_weighted_mean',
       'corbetta_language__atrophy_path_map_weighted_mean',
       'corbetta_motor__atrophy_path_map_weighted_mean',
       'corbetta_verbal_memory__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial_memory__atrophy_path_map_weighted_mean',
       'queensland_executive_q6__atrophy_path_map_weighted_mean',
       'queensland_language_q2__atrophy_path_map_weighted_mean',
       'queensland_visuospatial_q4__atrophy_path_map_weighted_mean'],
      dtype='object', length=204)

Regress out Covariate

In [122]:
# from calvin_utils.statistical_utils.regression_utils import RegressOutCovariates
# # use this code block to regress out covariates. Generally better to just include as covariates in a model..
# dependent_variable_list = lis
# regressors = ['Age', 'Sex']

# data_df, adjusted_dep_vars_list = RegressOutCovariates.run(df=data_df, dependent_variable_list=dependent_variable_list, covariates_list=regressors)
# print(adjusted_dep_vars_list)

Edit Column Values

In [123]:
data_df.columns

Index(['cohort_key', 'stimulation_modality', 'subject_id', 'subject',
       'delta_adasmem', 'percent_improvement_adas', 'age', 'baseline_adas',
       'followup_adas', 'atrophy_path',
       ...
       'queensland_visuospatial_q4__connectivity_path_map_weighted_mean',
       'corbetta_attention__atrophy_path_map_weighted_mean',
       'corbetta_language__atrophy_path_map_weighted_mean',
       'corbetta_motor__atrophy_path_map_weighted_mean',
       'corbetta_verbal_memory__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial_memory__atrophy_path_map_weighted_mean',
       'queensland_executive_q6__atrophy_path_map_weighted_mean',
       'queensland_language_q2__atrophy_path_map_weighted_mean',
       'queensland_visuospatial_q4__atrophy_path_map_weighted_mean'],
      dtype='object', length=204)

In [124]:
data_df.columns

Index(['cohort_key', 'stimulation_modality', 'subject_id', 'subject',
       'delta_adasmem', 'percent_improvement_adas', 'age', 'baseline_adas',
       'followup_adas', 'atrophy_path',
       ...
       'queensland_visuospatial_q4__connectivity_path_map_weighted_mean',
       'corbetta_attention__atrophy_path_map_weighted_mean',
       'corbetta_language__atrophy_path_map_weighted_mean',
       'corbetta_motor__atrophy_path_map_weighted_mean',
       'corbetta_verbal_memory__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial__atrophy_path_map_weighted_mean',
       'corbetta_visuospatial_memory__atrophy_path_map_weighted_mean',
       'queensland_executive_q6__atrophy_path_map_weighted_mean',
       'queensland_language_q2__atrophy_path_map_weighted_mean',
       'queensland_visuospatial_q4__atrophy_path_map_weighted_mean'],
      dtype='object', length=204)

In [125]:
from calvin_utils.file_utils.dataframe_utilities import convert_to_ordinal
# data_df, map = convert_to_ordinal(data_df, ['Frequency__Hz_', 'Stim_Relative_to_Memory_Cycle', 'Sensory_Domain', 'Location_Agg', 'Mean_Age', 'Disease', 'study'])

Misc Processing

In [126]:
# Drop rows where the absolute value of 'Pre_Post_Memory_Effect_Size__Cohen_s_D_' is greater than 4
# # mask = data_df['Pre_Post_Memory_Effect_Size__Cohen_s_D_'].abs() <= 4
# data_df = data_df[mask]
# data_df = data_df.reset_index(drop=True)
# data_df

# 02 - Define Your Formula

This is the formula relating outcome to predictors, and takes the form:
- y = B0 + B1 + B2 + B3 + . . . BN

It is defined using the columns of your dataframe instead of the variables above:
- 'Apples_Picked ~ hours_worked + owns_apple_picking_machine'

____
**ANOVA**
- Tests differences in means for one categorical variable.
- formula = 'Outcome ~ C(Group1)'

**2-Way ANOVA**
- Tests differences in means for two categorical variables without interaction.
- formula = 'Outcome ~ C(Group1) + C(Group2)'

**2-Way ANOVA with Interaction**
- Tests for interaction effects between two categorical variables.
- formula = 'Outcome ~ C(Group1) * C(Group2)'

**ANCOVA**
- Similar to ANOVA, but includes a covariate to control for its effect.
- formula = 'Outcome ~ C(Group1) + Covariate'

**2-Way ANCOVA**
- Extends ANCOVA with two categorical variables and their interaction, controlling for a covariate.
- formula = 'Outcome ~ C(Group1) * C(Group2) + Covariate'

**Multiple Regression**
- Assesses the impact of multiple predictors on an outcome.
- formula = 'Outcome ~ Predictor1 + Predictor2'

**Simple Linear Regression**
- Assesses the impact of a single predictor on an outcome.
- formula = 'Outcome ~ Predictor'

**MANOVA**
- Assesses multiple dependent variables across groups.
- Note: Not typically set up with a formula in statsmodels. Requires specialized functions.

____
Use the printout below to design your formula. 
- Left of the "~" symbol is the thing to be predicted. 
- Right of the "~" symbol are the predictors. 
- ":" indicates an interaction between two things. 
- "*" indicates and interactions AND it accounts for the simple effects too. 
- "+" indicates that you want to add another predictor. 

In [127]:
# data_df.columns
for column in data_df.columns:
    print(column)

cohort_key
stimulation_modality
subject_id
subject
delta_adasmem
percent_improvement_adas
age
baseline_adas
followup_adas
atrophy_path
connectivity_path
efield_path
percent_improvement_memory
memory_network_connectivity
hippocampus_connectivity
rios_network_connectivity
connectivity_damage
memory_network_damage
rios_network_damage
hippocampus_damage
stim_site_damage
data__schaefer50__parcel_01_A
data__schaefer50__parcel_01_FC
data__schaefer50__parcel_02_A
data__schaefer50__parcel_02_FC
data__schaefer50__parcel_03_A
data__schaefer50__parcel_03_FC
data__schaefer50__parcel_04_A
data__schaefer50__parcel_04_FC
data__schaefer50__parcel_05_A
data__schaefer50__parcel_05_FC
data__schaefer50__parcel_06_A
data__schaefer50__parcel_06_FC
data__schaefer50__parcel_07_A
data__schaefer50__parcel_07_FC
data__schaefer50__parcel_08_A
data__schaefer50__parcel_08_FC
data__schaefer50__parcel_09_A
data__schaefer50__parcel_09_FC
data__schaefer50__parcel_10_A
data__schaefer50__parcel_10_FC
data__schaefer50__par

In [ ]:
formula = "Month_12_Delta_ADASMemImprove ~ memory_network_connectivity * hippocampus_damage * stim_site_damage" #"*memory_network_damage + stim_site_damage" #memory_network_damage


# 02 - Visualize Your Design Matrix

This is the explanatory variable half of your regression formula
_______________________________________________________
Create Design Matrix: Use the create_design_matrix method. You can provide a list of formula variables which correspond to column names in your dataframe.

- design_matrix = palm.create_design_matrix(formula_vars=["var1", "var2", "var1*var2"])
- To include interaction terms, use * between variables, like "var1*var2".
- By default, an intercept will be added unless you set intercept=False
- **don't explicitly add the 'intercept' column. I'll do it for you.**

In [187]:
# Define the design matrix
outcome_matrix, design_matrix = cal_palm.define_design_matrix(formula, data_df, add_intercept=False)
design_matrix

Could not get a simple outcome and design matrix. Continuing with old code.


,memory_network_connectivity,hippocampus_damage,memory_network_connectivity:hippocampus_damage,stim_site_damage,memory_network_connectivity:stim_site_damage,hippocampus_damage:stim_site_damage,memory_network_connectivity:hippocampus_damage:stim_site_damage
1,-0.508993,-0.441218,0.224576,-0.323055,0.164433,0.142537,-0.072551
2,0.603874,-0.003571,-0.002157,0.249039,0.150388,-0.000889,-0.000537
4,0.929227,-0.910236,-0.845815,-0.821984,-0.763810,0.748199,0.695247
7,0.367548,-0.147347,-0.054157,-0.424454,-0.156007,0.062542,0.022987
8,0.617028,-0.835597,-0.515586,-1.005041,-0.620138,0.839809,0.518186
10,0.782354,-0.463981,-0.362998,-0.548393,-0.429037,0.254444,0.199065
11,0.730052,1.522639,1.111606,-0.808317,-0.590113,-1.230775,-0.898530
13,0.695987,-0.178131,-0.123977,0.119591,0.083233,-0.021303,-0.014826
15,0.551966,-0.100651,-0.055556,-0.650978,-0.359318,0.065522,0.036166
16,-2.383384,-0.273562,0.652003,0.524876,-1.250981,-0.143586,0.342221


# 03 - Visualize Your Dependent Variable

I have generated this for you based on the formula you provided

In [188]:
outcome_matrix

,Month_12_Delta_ADASMemImprove
1,0.733337
2,0.238952
4,0.733337
7,0.733337
8,-1.491392
10,1.969297
11,0.980529
13,0.486145
15,-0.502624
16,-0.749816


# 04 - Generate Contrasts

Generate a Contrast Matrix
- This is different from the contrast matrices used in cell-means regressions such as in PALM, but it is much more powerful. 



For more information on contrast matrices, please refer to this: https://cran.r-project.org/web/packages/codingMatrices/vignettes/codingMatrices.pdf

Generally, these drastically effect the results of ANOVA. However, they are mereley a nuisance for a regression.
In essence, they assess the coefficients of a given

________________________________________________________________
A coding matrix (a contrast matrix if it sums to zero) is simply a way of defining what coefficients to evaluate and how to evaluate them. 
If a coefficient is set to 1 and everything else is set to zero, we are taking the mean of the coefficient's means and assessing if they significantly
deviate from zero--IE we are checking if it had a significant impact on the ability to predict the depdendent variable.
If a coefficient is set to 1, another is -1, and others are 0, we are assessing how the means of the two coefficients deviate from eachother. 
If several coefficients are 1 and several others are -1, we are assessing how the group-level means of the two coefficients deviate from eachother.
If a group of coefficients are 1, a group is -1, and a group is 0, we are only assessing how the groups +1 and -1 have differing means. 

1: This value indicates that the corresponding variable's coefficient in the model is included in the contrast. It means you are interested in estimating the effect of that variable.

0: This value indicates that the corresponding variable's coefficient in the model is not included in the contrast. It means you are not interested in estimating the effect of that variable.

-1: This value indicates that the corresponding variable's coefficient in the model is included in the contrast, but with an opposite sign. It means you are interested in estimating the negative effect of that variable.

----------------------------------------------------------------
The contrast matrix is typically a matrix with dimensions (number of contrasts) x (number of regression coefficients). Each row of the contrast matrix represents a contrast or comparison you want to test.

For example, let's say you have the following regression coefficients in your model:

Intercept, Age, connectivity, Age_interaction_connectivity
A contrast matric has dimensions of [n_predictors, n_experiments] where each experiment is a contrast

If you want to test the hypothesis that the effect of Age is significant, you can set up a contrast matrix with a row that specifies this contrast (actually an averaging vector):
```
[0,1,0,0]. This is an averaging vector because it sums to 1
```
This contrast will test the coefficient corresponding to the Age variable against zero.


If you want to test the hypothesis that the effect of Age is different from the effect of connectivity, you can set up a contrast matrix with two rows:
```
[0,1,−1,0]. This is a contrast because it sums to 0
```

Thus, if you want to see if any given effect is significant compared to the intercept (average), you can use the following contrast matrix:
```
[1,0,0,0]
[-1,1,0,0]
[-1,0,1,0]
[-1,0,0,1] actually a coding matrix of averaging vectors
```

The first row tests the coefficient for Age against zero, and the second row tests the coefficient for connectivity against zero. The difference between the two coefficients can then be assessed.
_____
You can define any number of contrasts in the contrast matrix to test different hypotheses or comparisons of interest in your regression analysis.

It's important to note that the specific contrasts you choose depend on your research questions and hypotheses. You should carefully consider the comparisons you want to make and design the contrast matrix accordingly.

- Examples:
    - [Two Sample T-Test](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/GLM#Two-Group_Difference_.28Two-Sample_Unpaired_T-Test.29)
    - [One Sample with Covariate](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/GLM#Single-Group_Average_with_Additional_Covariate)

In [189]:
contrast_matrix = cal_palm.generate_basic_contrast_matrix(design_matrix)


Here is a basic contrast matrix set up to evaluate the significance of each variable.
Here is an example of what your contrast matrix looks like as a dataframe: 
   memory_network_connectivity  hippocampus_damage  \
0                            1                   0   
1                            0                   1   
2                            0                   0   
3                            0                   0   
4                            0                   0   
5                            0                   0   
6                            0                   0   

   memory_network_connectivity:hippocampus_damage  stim_site_damage  \
0                                               0                 0   
1                                               0                 0   
2                                               1                 0   
3                                               0                 1   
4                                               0 

Edit Contrast Matrix Here
- The generic contrast matrix will simply check if your Betas are significantly different from the intercept (average)

In [190]:
# # ptrn = oldhigh oldlow younghigh younglow
# contrast_matrix = [
#     [0, 0, 0, 0],
#     [0, 1, 0, 0],
#     [0, 0, 1, 0],
#     [0, 0, 0, 1],
# ]

Finalize Contrast Matrix

In [191]:
contrast_matrix_df = cal_palm.finalize_contrast_matrix(design_matrix=design_matrix, 
                                                    contrast_matrix=contrast_matrix) 
contrast_matrix_df

,memory_network_connectivity,hippocampus_damage,memory_network_connectivity:hippocampus_damage,stim_site_damage,memory_network_connectivity:stim_site_damage,hippocampus_damage:stim_site_damage,memory_network_connectivity:hippocampus_damage:stim_site_damage
0,1,0,0,0,0,0,0
1,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0
3,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0
5,0,0,0,0,0,1,0
6,0,0,0,0,0,0,1


# 06 - Run the Regression

Regression Results Are Displayed Below

In [192]:
import statsmodels.api as sm
# Fit the regression model
model = sm.OLS(outcome_matrix, design_matrix)
results = model.fit()
results.summary2()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                                       Results: Ordinary least squares
==============================================================================================================
Model:                          OLS                                 Adj. R-squared (uncentered):       0.032  
Dependent Variable:             Month_12_Delta_ADASMemImprove       AIC:                               90.1885
Date:                           2026-05-13 10:05                    BIC:                               99.9969
No. Observations:               30                                  Log-Likelihood:                    -38.094
Df Model:                       7                                   F-statistic:                       1.142  
Df Residuals:                   23                                  Prob (F-statistic):                0.372  
R-squared (uncentered):         0.258                               Scale:                             0.96797
--------------------------------------------------------------------------------------------------------------
                                                                 Coef.  Std.Err.    t    P>|t|   [0.025 0.975]
--------------------------------------------------------------------------------------------------------------
memory_network_connectivity                                      0.2662   0.2237  1.1901 0.2461 -0.1965 0.7289
hippocampus_damage                                              -0.3171   0.2754 -1.1514 0.2614 -0.8869 0.2526
memory_network_connectivity:hippocampus_damage                   0.3199   0.3317  0.9644 0.3449 -0.3663 1.0061
stim_site_damage                                                 0.0347   0.2528  0.1374 0.8919 -0.4883 0.5577
memory_network_connectivity:stim_site_damage                     0.1696   0.3567  0.4756 0.6388 -0.5682 0.9075
hippocampus_damage:stim_site_damage                              0.2054   0.1887  1.0883 0.2877 -0.1850 0.5957
memory_network_connectivity:hippocampus_damage:stim_site_damage -0.3930   0.1930 -2.0362 0.0534 -0.7923 0.0063
--------------------------------------------------------------------------------------------------------------
Omnibus:                              0.217                       Durbin-Watson:                         2.012
Prob(Omnibus):                        0.897                       Jarque-Bera (JB):                      0.370
Skew:                                 -0.166                      Prob(JB):                              0.831
Kurtosis:                             2.569                       Condition No.:                         7    
==============================================================================================================
Notes:
[1] R² is computed without centering (uncentered) since the                 model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Visualize the Regression as a Plane

In [193]:
from calvin_utils.statistical_utils.response_surface_utils.regression_response_surface import GLMPlot
ip = GLMPlot(model_results=results, data_df=data_df, formula=formula)
ip.run(plot_residuals=True)

ValueError encountered. Refitting the model with the provided formula.
(30,)
(30,)


Visualize the Regression as a Forest Plot
- This will probably look poor if you ran a regression without standardizing your data. 

In [ ]:
from calvin_utils.statistical_utils.statistical_measurements import ForestPlot
forest = ForestPlot(model=results, sig_digits=2, out_dir=out_dir, table=False)
forest.run()

Visualize The Model's Fit

In [ ]:
from calvin_utils.statistical_utils.statistical_measurements import model_diagnostics
model_diagnostics(results)

Visualize the Partial Regression Plots

In [ ]:
from calvin_utils.statistical_utils.statistical_measurements import PartialRegressionPlot
partial_plot = PartialRegressionPlot(model=results, design_matrix=design_matrix, out_dir='/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/cognition_2023/post-publication_analyses/partial_regressoin', palette='Reds')
partial_plot = partial_plot.run()
# 

In [ ]:
partial_plot.run_scattergrid()

2D visualization for regression

In [ ]:
print(formula)

In [ ]:
from calvin_utils.statistical_utils.anova_visualization import QuickRegressionMarginalPlot
import statsmodels.formula.api as smf

model = smf.ols(formula, data=data_df).fit()

plotter = QuickRegressionMarginalPlot(model, data_df, out_dir=out_dir)
plotter.plot_predictions(
    x_var="Subiculum_Connectivity_T_Redone",
    y_var="Percent_Cognitive_Improvement",          # optional; inferred from formula if omitted
    hue_var=None,    # optional
    cohort_var=None,  # optional
    stdevs=[0, -3],  # default [0] = mean of other continuous vars
    include_ci=True     # optional
)


2D visualization for categorical (anova)

In [ ]:
print(formula)

In [ ]:
from calvin_utils.statistical_utils.anova_visualization import QuickANOVAPlot
import statsmodels.formula.api as smf
model = smf.ols(formula, data=data_df).fit()
quick_plot = QuickANOVAPlot(model, data_df, out_dir=out_dir)
quick_plot.make_predictions()
quick_plot.plot_predictions(x_var='Age', hue_var='Dataset', cohort_var='City', y='z_scored_improPercent_Cognitive_Improvementvement', error_bar='se')

# 07 - Run the Contrasts

Contrast Results Are Displayed Below

In [ ]:
contrast_matrix_df

In [ ]:
contrast_results = results.t_test(contrast_matrix_df)
print(contrast_results.summary())

# 08 - Compare the Coefficient Between 2 Groups

In [ ]:
groups_column = 'Age_Group'

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import RegressionAnalysis
regression_test = RegressionAnalysis(outcome_df=outcome_matrix, design_df=design_matrix, groups_df=data_df[[groups_column]], N=10000, metric='similarity', two_tail=False, out_dir=out_dir)
regression_test.run()

# 09 - Compare Distribution of T Values Between Groups

In [ ]:
data_df.columns

In [ ]:
plot_together=False
groups_column = 'Age_Group'

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import BootstrappedRegressionAnalysis

# Create an instance of BootstrappedRegressionAnalysis
bootstrapped_regression_test = BootstrappedRegressionAnalysis(outcome_df=outcome_matrix, design_df=design_matrix, groups_df=data_df[[groups_column]], N=10000, out_dir=out_dir, plot_together=plot_together)

# Run the bootstrapped regression analysis
df1 = bootstrapped_regression_test.run()

# 10a - Similarity of Response topology Across Cohorts in Dataframe
- Denote the column containing your different cohorts to test across
- Write the formula, excluding the cohort column

In [ ]:
cohort_col = 'City'
formula = "Z_Scored_Percent_Cognitive_Improvement ~ Age*Z_Scored_Subiculum_Connectivity_T"

In [ ]:
from calvin_utils.statistical_utils.response_surface_utils.permute_response_surfaces import GLMPredictionPermutationTest
sim_test = GLMPredictionPermutationTest(data_df=data_df, formula=formula, cohort_col=cohort_col, two_tailed=False, method='pearsonr',
                                        n_permutations=10000)
sim_test.run()

# 10b - Predict Another Dataframe
- Can use this to predict data from a second group, such as the 'other_df' defined in "Step 01, Drop Rows Based on Value of a Column"

In [ ]:
import numpy as np
from scipy.stats import f

def calculate_ssr(observations, predictions):
    """
    Calculate the regression sum of squares.
    This is the sum of squared deviations, with deviation being Y_hat_i - Y_bar
    
    SSR is a measure used to quantify the variance in the observed data that is not explained by the model. 
    It is calculated as the sum of the squares of the differences between the observed values and the model's predictions. 
    The more the 'mean' predicts 
    A lower SSR indicates a better model fit, meaning the model's predictions are closer to the actual observations.
    
    SSR = Σ(y_hat - y_bar)^2 
    
    Parameters:
    - observations (array-like): The actual observed outcomes.
    - predictions (array-like): The outcomes predicted by the model.
    
    Returns:
    - float: The calculated SSR.
    """
    y_hat = observations
    y_bar = np.mean(predictions)
    ssr = np.sum((y_hat - y_bar) ** 2)
    return ssr

def calculate_sse(observations, predictions):
    """
    Calculate the sum of squares due to error (SSE).
    
    SSE is a measure of the total deviation of the response values from the fit to the response values. 
    It is calculated as the sum of the squares of the differences between the predicted values and the observed values. 
    A lower SSE indicates a model that more accurately fits the data.
    
    SSE = Σ(y - y_hat)^2
    
    Parameters:
    - observations (array-like): The actual observed outcomes.
    - predictions (array-like): The outcomes predicted by the model.
    
    Returns:
    - float: The calculated SSE.
    """
    y = observations
    y_hat = predictions
    sse = np.sum((y - y_hat) ** 2)
    return sse

def calculate_ssto(observations):
    """
    Calculate the total sum of squares (SSTO).
    
    SSTO is a measure of the total variance in the observed data and is used as a comparative tool for model evaluation. 
    It is calculated as the sum of the squares of the differences between the observed values and their overall mean. 
    SSTO is used in the denominator of the coefficient of determination, R^2, which assesses the fit of the model.
    
    SSTO = Σ(y - y_bar)^2
    
    Parameters:
    - observations (array-like): The actual observed outcomes.
    
    Returns:
    - float: The calculated SSTO.
    """
    y = observations
    y_bar = np.mean(y)
    ssto = np.sum((y - y_bar) ** 2)
    return ssto


def calculate_msr(ssr, num_regressors):
    """
    Calculate the mean square due to regression (MSR).
    
    MSR is a measure of the variation explained by the independent variables in the model. It is calculated as the 
    sum of squared residuals (SSR) divided by the degrees of freedom, which is the number of independent variables (regressors) minus one.
    A higher MSR indicates that the model explains a greater amount of variation in the outcome variable.
    
    MSR = SSR / (number of regressors - 1)
    
    Parameters:
    - ssr (float): The sum of squared residuals from the regression model.
    - num_regressors (int): The number of independent variables in the model.
    
    Returns:
    - float: The calculated MSR.
    """
    return ssr / (num_regressors - 1)

def calculate_mse(sse, num_regressors, num_observations):
    """
    Calculate the mean square error (MSE).
    
    MSE is a measure of the average of the squares of the errors, that is, the average squared difference between the observed actual outcomes and the outcomes predicted by the model. It is calculated as the sum of squared errors (SSE) divided by the degrees of freedom, which is the number of observations minus the number of regressors.
    A lower MSE indicates a better fit of the model to the data.
    
    MSE = SSE / (number of observations - number of regressors)
    
    Parameters:
    - sse (float): The sum of squared errors from the regression model.
    - num_regressors (int): The number of independent variables in the model.
    - num_observations (int): The number of observations in the data set.
    
    Returns:
    - float: The calculated MSE.
    """
    return sse / (num_observations - num_regressors)

def calculate_f_stat(msr, mse, num_regressors, num_observations):
    """
    Calculate the F-statistic.
    
    The F-statistic is used to compare statistical models that have been fitted to a data set in order to identify the model that best fits the population from which the data were sampled. It is the ratio of the mean square due to regression (MSR) to the mean square error (MSE).
    
    The F-statistic follows the F-distribution under the null hypothesis that the model with no independent variables fits the data as well as your model. A higher F-statistic implies that the null hypothesis is false, and your model adds value in explaining the variation in the data.
    
    F = MSR / MSE
    
    Parameters:
    - msr (float): The mean square due to regression.
    - mse (float): The mean square error.
    
    Returns:
    - float: The calculated F-statistic.
    - float: The p-value from the F-distribution.
    """
    f_stat = msr / mse
    # The degrees of freedom for the numerator (dfn) is the number of independent variables (regressors).
    # The degrees of freedom for the denominator (dfd) is the total number of observations minus the number of independent variables minus 1.
    # These values need to be defined or calculated outside of this function.
    dfn = num_regressors - 1
    dfd = num_observations - num_regressors 
    p_value = f.sf(f_stat, dfn, dfd)
    return f_stat, p_value

def run_goodness_of_fit(target_outcome_matrix, predictions, target_design_matrix):
    """
    Calculate the F-statistic and p-value for a linear regression model.

    Parameters:
    - target_outcome_matrix (array-like): The actual observed outcomes (Y_actual).
    - predictions (array-like): The outcomes predicted by the model (Y_hat).

    Returns:
    - float: The calculated F-statistic.
    - float: The p-value from the F-distribution.
    """
    # Calculate the regression sum of squares (SSR).
    ssr = calculate_ssr(target_outcome_matrix, predictions)

    # Calculate the sum of squares due to error (SSE).
    sse = calculate_sse(target_outcome_matrix, predictions)

    # Calculate the number of regressors and observations.
    num_regressors = target_design_matrix.shape[1]
    num_observations = len(target_outcome_matrix)

    # Calculate the mean square due to regression (MSR).
    msr = calculate_msr(ssr, num_regressors)

    # Calculate the mean square error (MSE).
    mse = calculate_mse(sse, num_regressors, num_observations)

    # Calculate the F-statistic and p-value.
    f_stat, p_value = calculate_f_stat(msr, mse, num_regressors, num_observations)

    return f_stat, p_value

def calculate_r_squared(observations, predictions):
    """
    Calculate the R-squared (coefficient of determination) value.

    R-squared measures the proportion of the variance in the observed outcomes that is explained by the predictions.

    R-squared = 1 - (SSE / SSTO)

    Parameters:
    - observations (array-like): The actual observed outcomes.
    - predictions (array-like): The outcomes predicted by the model.

    Returns:
    - float: The calculated R-squared value.
    """
    # Calculate SSE (Sum of Squares of Errors)
    sse = np.sum((observations - predictions) ** 2)

    # Calculate SSTO (Total Sum of Squares)
    y_mean = np.mean(observations)
    ssto = np.sum((observations - y_mean) ** 2)

    # Calculate R-squared
    r_squared = 1 - (sse / ssto)
    
    return r_squared

Get a new Dataframe to predict upon

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()

In [ ]:
column = 'Cohort'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 0 # The value to drop if T

In [ ]:
data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

In [ ]:
# Define the design matrix
target_outcome_matrix, target_design_matrix = cal_palm.define_design_matrix(formula, data_df)
predictions = results.predict(target_design_matrix)

Extract F-Test for Goodness of Fit

In [ ]:
f_stat, p_value = run_goodness_of_fit(target_outcome_matrix=target_outcome_matrix.to_numpy().flatten(), 
                    predictions=predictions.to_numpy().flatten(), 
                    target_design_matrix=target_design_matrix)
print(p_value)

Extract R-Squared

In [ ]:
from scipy.stats import pearsonr
r_squared = calculate_r_squared(target_outcome_matrix.to_numpy().flatten(), predictions.to_numpy().flatten())
print(f"R-squared: {r_squared:.4f}")
r, p = pearsonr(target_outcome_matrix.to_numpy().flatten(), predictions.to_numpy().flatten())
print(f"PearsonR R-squared: {r*r}")

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(target_outcome_matrix, predictions)

# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Calculate Mean Absolute Error (MAE)
mae = mean_absolute_error(target_outcome_matrix, predictions)

# Print the results
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)

Plot The Fit

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_scatter_with_f_stat_in_title(target_outcome_matrix, predictions, f_stat, p_value, out_dir=None):
    """
    Create a scatterplot of predicted vs. observed values with F-statistic and p-value in the title.

    Parameters:
    - target_outcome_matrix (array-like): The actual observed outcomes.
    - predictions (array-like): The outcomes predicted by the model.
    - f_stat (float): The F-statistic value.
    - p_value (float): The p-value.

    Returns:
    - None (displays the plot)
    """
    # Create a scatterplot using Seaborn
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=predictions,
                    y=target_outcome_matrix,
                    label='Predicted vs. Observed')

    # Add a diagonal line
    xlim = plt.xlim()  # Get current X-axis limits
    ylim = plt.ylim()  # Get current Y-axis limits
    min_limit = min(xlim[0], ylim[0])
    max_limit = max(xlim[1], ylim[1])
    plt.plot([min_limit, max_limit], [min_limit, max_limit], linestyle='--', color='gray', label='Perfect Fit')

    # Add labels and title with F-statistic and p-value
    plt.xlabel('Predicted Values')
    plt.ylabel('Observed Values')
    plt.title(f'Scatterplot of Predicted vs. Observed Values\nF-statistic: {f_stat:.2f}, p-value: {p_value:.4f}')

    # Set axis limits
    plt.xlim(min_limit, max_limit)
    plt.ylim(min_limit, max_limit)

    # Show legend
    plt.legend()
    
    if out_dir:
        # Save the figure
        plt.savefig(f"{out_dir}/predicted_plot.png", bbox_inches='tight')
        plt.savefig(f"{out_dir}/predicted_plot.svg", bbox_inches='tight')
        print(f'Saved to {out_dir}/predicted_plot.svg')

    # Display the plot
    plt.grid()
    plt.show()

def plot_residuals(target_outcome_matrix, predictions, f_stat, p_value, out_dir=None):
    """
    Create a scatterplot of residuals with F-statistic and p-value in the title and save it.

    Parameters:
    - target_outcome_matrix (array-like): The actual observed outcomes.
    - predictions (array-like): The outcomes predicted by the model.
    - f_stat (float): The F-statistic value.
    - p_value (float): The p-value.
    - out_dir (str, optional): The directory to save the plot. If None, the plot won't be saved.

    Returns:
    - None (displays the plot)
    """
    # Calculate residuals
    residuals = target_outcome_matrix - predictions

    # Create a scatterplot of residuals
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=predictions,
                    y=residuals,
                    label='Residuals vs. Predicted')

    # Calculate y-axis limits
    y_lim_min = min(-3, np.min(residuals) - 0.5)
    y_lim_max = max(3, np.max(residuals) + 0.5)

    # Set y-axis limits
    plt.ylim(y_lim_min, y_lim_max)

    # Add a horizontal line at y=0
    plt.axhline(0, color='gray', linestyle='--', label='Zero Residual')

    # Add labels and title with F-statistic and p-value
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    plt.title(f'Scatterplot of Residuals vs. Predicted Values\nF-statistic: {f_stat:.2f}, p-value: {p_value:.4f}')

    # Show legend
    plt.legend()

    if out_dir:
        # Save the figure
        plt.savefig(f"{out_dir}/residuals_plot.png", bbox_inches='tight')
        plt.savefig(f"{out_dir}/residuals_plot.svg", bbox_inches='tight')
        print(f'Saved to {out_dir}/residuals_plot.svg')

    # Display the plot
    plt.grid()
    plt.show()

Residuals Plot

In [ ]:
plot_residuals(target_outcome_matrix.to_numpy().flatten(), predictions.to_numpy().flatten(), f_stat, p_value, out_dir)

In [ ]:
plot_scatter_with_f_stat_in_title(target_outcome_matrix.to_numpy().flatten(), predictions.to_numpy().flatten(), f_stat, p_value, out_dir)


# 11 - Visualize 2-Way ANOVA (Model-Free)

Generate an Interaction Plot Based on the Raw Data (Model-Free)
- data_df (pd.DataFrame): DataFrame containing the data.
- group1_column (str): Name of the first grouping column (categorical).
- group2_column (str): Name of the second grouping column (categorical).
- outcome_column (str): Name of the outcome column (continuous).

In [ ]:
formula

In [ ]:
data_df=data_df
group_1='sbc_conn'
group_2='age'
outcome_column='Dataset'

In [ ]:
from calvin_utils.statistical_utils.anova_visualization import ModelFreeInteractionPlot
summary_df = ModelFreeInteractionPlot.calculate_means_and_sem(data_df, group_1, group_2, outcome_column)
# Ensure Data is Okay
ModelFreeInteractionPlot.diagnose_data(data_df, group_1, group_2, outcome_column)
# Plot
ModelFreeInteractionPlot.plot_interaction_error_bar(summary_df, group_1, group_2, out_dir=out_dir)
# Test for differences in group_1 at each level of group_2. 
print(ModelFreeInteractionPlot.perform_kruskal_wallis_test(data_df, group_1, group_2, outcome_column))
# Test for significant differences 
print(ModelFreeInteractionPlot.perform_contrast(data_df, contrast_column=group_1, outcome_column=outcome_column))

# Visualize a Modelled ANOVA (Predictions upon Existing Data Points)
- The class expects the a dataframe containing all of the columns with your regressors (or more columns)
- fit a formula onto a model
    - Example: 'Z_Scored_Percent_Cognitive_Improvement ~ Age_Group*Subiculum_Group_By_24 + City'
- Then, you must plot model's predictions of the observed data
    - Mandatory:
        - x_var: the factors
        - hue_var: the levels
    - Optional:
        - cohort_var: additional traces for cohorts
        - y: defaults to 'predictions', and sets the y-axis to the estimated marginal means of the model. Can set to another column in data_df. 
- Bars represent 95%CI
- note: you do not need the input variables to be part of the dataframe. You can model predictions then split by other variables!

In [ ]:
data_df.columns

Visualize as a Bar Plot

In [ ]:
quick_plot.plot_bar(x_var='Subiculum_Group_By_Inflection_Point', hue_var='City', cohort_var=None, y='predictions', error_bar=None)

Read the Regression Results

In [ ]:
print(model.summary2())

# 12 - Visualize a GLM's Predictions Across a Range of Potential Data (AKA Profile Plot or Marginal Plot)
- **this is pretty complex stuff, so be sure to read up on Marginals**
- This is a plot which generates an estimated marginal estimate across a set number of categories/factors
    - This is similar to a marginal mean, but a marginal mean provides the 
- Do **not** set any values to a list of strings. If you want to use categorical data, encode the categories as ordinal values and re-run the model. Then set the categories in the values.

In [ ]:
formula

In [ ]:
marginal_scenarios_dict = {'Age_Group': [0, 1],'City':[0, 1], 'Z_Scored_Subiculum_Connectivity_T':['continuous']}

In [ ]:
from calvin_utils.statistical_utils.statistical_measurements import GLMMarginalsPlot
factor_plot = GLMMarginalsPlot(formula, data_df, model=results, data_range=None, marginal_scenarios_dict=marginal_scenarios_dict, variance_bars=None, out_dir=out_dir)
factor_plot.run()

# Visualize a GLM's Predictions at Points in Data (Estimated Marginal Means)

Interaction Plot

Identify the column names used to generate the original formula. The order of the list determines how variables are categorized in the following interaction plot. 
- This expects the formula to have defined categorical variables, like:
    - Z_Scored_Percent_Cognitive_Improvement ~ C(Subiculum_Group_By_24)*Age + C(City)

In [ ]:
formula

In [ ]:
def create_interaction_plot_dynamic(emm_df, categorical_vars, out_dir=None):
    """
    Create an interaction plot for 2 or 3 categorical variables in emm_df.

    Parameters:
    - emm_df: DataFrame containing the data
    - categorical_vars: List of strings specifying the categorical variables
    - ms: Marker size for the plot (default is 10)
    """
    # Check if we have 2 or 3 categorical variables
    if len(categorical_vars) == 2:
        plot_interaction_2_vars(emm_df, categorical_vars, out_dir)
    elif len(categorical_vars) == 3:
        self.plot_interaction_3_vars(emm_df, categorical_vars,out_dir)
    else:
        print("This function supports only 2 or 3 categorical variables.")
        

import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns

def plot_interaction_2_vars(emm_df, categorical_vars, output_directory=None, color_list = None):
    """
    Plots an interaction bar plot for two categorical variables in an ANOVA setup.

    Parameters:
        emm_df (DataFrame): The DataFrame containing the data.
        categorical_vars (list): A list of two categorical variable names to plot.
        color_list (list, optional): List of colors for the bars. If None, defaults to tab10 colormap.
        output_directory (str, optional): Directory to save the plot. If None, plot is not saved.
    """
    # Set up the color palette
    if color_list is None:
        color_palette = sns.color_palette("tab10", len(emm_df[categorical_vars[1]].unique()))
    else:
        color_palette = sns.color_palette(color_list, len(emm_df[categorical_vars[1]].unique()))

    # color_palette = sns.color_palette("tab10")
    
    # Initialize the plot
    plt.figure(figsize=(10, 8))

    # Create the bar plot
    sns.barplot(
        data=emm_df,
        x=categorical_vars[0],
        y='predictions',
        hue=categorical_vars[1],
        palette=color_palette
    )

    # Customize the plot
    plt.xlabel(categorical_vars[0])
    plt.ylabel('Predictions')
    plt.title(f'Interaction Bar Plot of Predicted EMMs ({categorical_vars[1]})')
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save the plot if output_directory is provided
    if output_directory:
        plt.savefig(f"{output_directory}/interaction_bar_plot.png", bbox_inches='tight')
        plt.savefig(f"{output_directory}/interaction_bar_plot.svg", bbox_inches='tight')
        print(f'Saved to {output_directory}/interaction_bar_plot.png and {output_directory}/interaction_bar_plot.svg')

    # Show the plot
    plt.show()

In [ ]:
from calvin_utils.statistical_utils.statistical_measurements import EstimatedMarginalMean

interaction_plot = EstimatedMarginalMean(formula, data_df, model=results)
interaction_plot.extract_unique_variables()
interaction_plot.create_emm_df(plus_2_stdev=True, minus_2_stdev=False)
interaction_plot.define_design_matrix()
interaction_plot.predict_emm_design_df()
categorical_columns = interaction_plot.variables_df[interaction_plot.variables_df['Type'] == 'categorical']['Variable'].tolist()

create_interaction_plot_dynamic(emm_df=interaction_plot.emm_df.dropna(), categorical_vars=categorical_columns, out_dir=out_dir)

In [ ]:
interaction_plot.emm_df

# 12 - Get Zero Points of Each Coefficient

The zero point is the point where the coefficient cross zero.
When the response topology has a saddle point, the zero point indicates where the saddle is. 
Technically, the linear regression's formula does not have a saddle point. 
However, if you rotate the response topology which exhibits a saddle point by 45 degrees, you will observe a saddle point in orientation with the Cartesian plane. 

In [ ]:
from calvin_utils.statistical_utils.calculus_utils import find_zero_point_of_coefficients
find_zero_point_of_coefficients(results)

# 13 - Compare 2 Different Sets of Regressors

In [ ]:
data_df.columns

In [ ]:
formula

For Nested Models

In [82]:
from statsmodels.stats.api import anova_lm
import statsmodels.formula.api as smf

smaller_formula = 'Month_12_Delta_ADASMemImprove ~ overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas + stim_site_damage'

larger_formula = 'Month_12_Delta_ADASMemImprove ~ overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas * stim_site_damage'
#----------------------------------------------------------------DO NOT TOUCH!----------------------------------------------------------------
table1 = anova_lm(smf.ols(smaller_formula, data=data_df).fit(), smf.ols(larger_formula, data=data_df).fit())
print(table1)

   df_resid        ssr  df_diff   ss_diff         F    Pr(>F)
0      27.0  24.599129      0.0       NaN       NaN       NaN
1      26.0  19.859797      1.0  4.739333  6.204628  0.019453


For Un-nested Models
- Need to employ permutation because there is no existing method. 

In [70]:
for column in data_df.columns:
    print(column)

cohort_key
stimulation_modality
subject_id
subject
delta_adasmem
percent_improvement_adas
age
baseline_adas
followup_adas
atrophy_path
connectivity_path
efield_path
percent_improvement_memory
memory_network_connectivity
hippocampus_connectivity
rios_network_connectivity
connectivity_damage
memory_network_damage
rios_network_damage
hippocampus_damage
stim_site_damage
data__schaefer50__parcel_01_A
data__schaefer50__parcel_01_FC
data__schaefer50__parcel_02_A
data__schaefer50__parcel_02_FC
data__schaefer50__parcel_03_A
data__schaefer50__parcel_03_FC
data__schaefer50__parcel_04_A
data__schaefer50__parcel_04_FC
data__schaefer50__parcel_05_A
data__schaefer50__parcel_05_FC
data__schaefer50__parcel_06_A
data__schaefer50__parcel_06_FC
data__schaefer50__parcel_07_A
data__schaefer50__parcel_07_FC
data__schaefer50__parcel_08_A
data__schaefer50__parcel_08_FC
data__schaefer50__parcel_09_A
data__schaefer50__parcel_09_FC
data__schaefer50__parcel_10_A
data__schaefer50__parcel_10_FC
data__schaefer50__par

In [62]:
formula

'Month_12_Delta_ADASMemImprove ~ overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas * memory_network_damage + stim_site_damage'

In [76]:
formula1 = 'Month_12_Delta_ADASMemImprove ~ overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas'
formula2 = 'Month_12_Delta_ADASMemImprove ~ overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas * overlap_stimpyper_optimized_bilateral_vtas_vs_stimpyper_clinician_bilateral_vtas'

In [77]:
import numpy as np
from tqdm import tqdm
import statsmodels.formula.api as smf

# Original R^2 values from the models
r2_model1 = smf.ols(formula1, data=data_df).fit().rsquared
r2_model2 = smf.ols(formula2, data=data_df).fit().rsquared

# Difference in R of original models
original_diff = r2_model1 - r2_model2

# Number of permutations
n_permutations = 1000

# Store differences from permutations
perm_diffs = []
for i in tqdm(range(n_permutations)):
    # Permute the outcome variable
    data_df_permuted = data_df.copy()
    data_df_permuted['Month_12_Delta_ADASMemImprove'] = np.random.permutation(data_df_permuted['Month_12_Delta_ADASMemImprove'].values)
    
    # Fit the models to the permuted dataset and calculate R^2
    perm_model1 = smf.ols(formula1, data=data_df_permuted).fit()
    perm_model2 = smf.ols(formula2, data=data_df_permuted).fit()
    
    # Difference in R^2 for the permuted models
    perm_diff = perm_model1.rsquared - perm_model2.rsquared
    
    # Store the difference
    perm_diffs.append(perm_diff)

# Calculate the p-value as the proportion of permuted differences
# that are greater than or equal to the observed difference
p_value = np.mean([abs(diff) >= abs(original_diff)for diff in perm_diffs])
print(f"Original R Difference: {original_diff}")
print(f"P-value from permutation test: {p_value}")

100%|██████████| 1000/1000 [00:03<00:00, 318.15it/s]

Original R Difference: 0.0
P-value from permutation test: 1.0


In [68]:
r2_model1

0.17581331649524612

In [ ]:
np.sqrt(.30)

In [ ]:
r2_model2

Estimate Uncertainty Around a Regression Metric with Bootstrapping
- R-squared approach visualized below. 

In [ ]:
#Testing
import numpy as np
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
from sklearn.utils import resample

def bootstrap_r_squared(data: pd.DataFrame, formula: str, n_bootstraps: int = 1000):
    """
    Perform bootstrap resampling to calculate the 95% confidence interval for R-squared.

    Parameters:
    - data (pd.DataFrame): The primary dataframe containing the data.
    - formula (str): The formula specifying the model (e.g., 'Y ~ X1 + X2').
    - n_bootstraps (int): The number of bootstrap samples (default is 10,000).

    Returns:
    - dict: A dictionary containing the original R-squared, the mean R-squared, the 95% confidence interval,
            the standard deviation, and the standard error of R-squared.
    """
    # Original model
    model = sm.OLS.from_formula(formula, data)
    original_results = model.fit()
    print("Original R^2:", original_results.rsquared)

    # Bootstrapping
    r_squared_values = []
    for _ in tqdm(range(n_bootstraps)):
        # Resample the dataframe with replacement
        boot_data = resample(data, replace=True, n_samples=len(data))
        
        # Fit the model to the bootstrapped sample
        boot_model = sm.OLS.from_formula(formula, boot_data)
        boot_results = boot_model.fit()
        
        # Store the R-squared
        r_squared_values.append(boot_results.rsquared)

    # Calculate the 95% confidence interval for R^2
    r_squared_lower = np.percentile(r_squared_values, 2.5)
    r_squared_upper = np.percentile(r_squared_values, 97.5)

    result = {
        "Original R^2": original_results.rsquared,
        "Mean R^2": np.mean(r_squared_values),
        "95% CI": (r_squared_lower, r_squared_upper),
        "Stdev": np.std(r_squared_values),
        "StdErr R^2": np.std(r_squared_values) / np.sqrt(len(r_squared_values))
    }

    print(f"95% CI for R^2: ({r_squared_lower:.4f}, {r_squared_upper:.4f})")
    print(f"Mean R^2: {np.mean(r_squared_values)} | Stdev: {np.std(r_squared_values)} | StdErr R^2: {np.std(r_squared_values)/np.sqrt(len(r_squared_values))}")

    return result

bootstrap_r_squared(data_df, formula5)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.utils import resample

# Number of bootstrap samples
n_bootstraps = 10000
r_squared_values = []

# Original model
model = sm.OLS(outcome_matrix, design_matrix)
original_results = model.fit()
print("Original R^2:", original_results.rsquared)

# Bootstrapping
for _ in range(n_bootstraps):
    # Resample the indices with replacement
    boot_indices = np.random.choice(len(outcome_matrix), size=len(outcome_matrix), replace=True)
    if isinstance(outcome_matrix, pd.DataFrame):
        boot_outcome = outcome_matrix.iloc[boot_indices]
    else:
        boot_outcome = outcome_matrix[boot_indices]
    
    if isinstance(design_matrix, pd.DataFrame):
        boot_design = design_matrix.iloc[boot_indices]
    else:
        boot_design = design_matrix[boot_indices]

    # Fit the model to the bootstrapped sample
    boot_model = sm.OLS(boot_outcome, boot_design)
    boot_results = boot_model.fit()
    
    # Store the R-squared
    r_squared_values.append(boot_results.rsquared)

# Calculate the 95% confidence interval for R^2
r_squared_lower = np.percentile(r_squared_values, 2.5)
r_squared_upper = np.percentile(r_squared_values, 97.5)

print(f"95% CI for R^2: ({r_squared_lower:.4f}, {r_squared_upper:.4f})")
print(f"Mean R^2: {np.mean(r_squared_values)} | Stdev: {np.std(r_squared_values)} | StdErr R^2: {np.std(r_squared_values)/len(r_squared_values)}")

Test the means

In [ ]:
import numpy as np
from scipy.stats import t

def two_sample_t_test(n1, n2, mean1, mean2, stdev1, stdev2):
    t_stat = (mean1 - mean2) / np.sqrt((stdev1**2/n1 + stdev2**2/n2))
    
    dof = n1 + n2 - 2
    p_value = 2 * (1 - t.cdf(np.abs(t_stat), dof))
    return t_stat, p_value

t_stat, p_value = two_sample_t_test(n1=10, n2=10, mean1=0.16, mean2=0.23, stdev1=0.09, stdev2=0.09)
print("t-statistic:", t_stat)
print("p-value:", p_value)


In [ ]:
np.sqrt(.11)